In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Fritsche Neto Plant Breeding Program - AlphaSimPy Tutorial Notebook

This notebook converts the provided BRAID breeding program abstraction into a runnable AlphaSimPy-style workflow.

It demonstrates a simplified heterotic-pool hybrid breeding program with:
- C0 heterotic pools A and B
- line derivation from each pool
- testcross hybrid creation using three testers per side
- phenotypic evaluation of hybrids
- selection of superior hybrids and parent pools
- recombination to form C1 heterotic pools
- simple product development and release placeholders

**Source**: BRAID abstraction provided by user  
**Package**: AlphaSimPy  
**Style**: Tutorial notebook modeled after AlphaSimPy examples

## BRAID Interpretation and Assumptions

The BRAID abstraction specifies the program logic clearly, but several operational values were marked as variable or unspecified.

To make the notebook executable, the following explicit assumptions are used:

1. Each C0 heterotic pool starts with 60 inbred founder individuals.
2. One cycle corresponds to one recurrent selection round.
3. Line derivation is approximated with one generation of random crossing followed by phenotypic evaluation and selection, rather than a full CMS/RGA/restorer implementation.
4. Three testers are sampled from the opposite heterotic pool in each cycle.
5. Each selected line is crossed to all three opposite-side testers to create testcross hybrids.
6. Hybrid performance is modeled as a single additive trait with heritability approximated through `varE`.
7. The top 10% of evaluated hybrids are advanced to product development.
8. Parent recombination into C1 pools is approximated by selecting the parental lines that contributed to top hybrids and randomly recombining them.
9. Product development and release are represented as selected hybrid sets rather than a full multi-stage advancement pipeline.

These assumptions are intentionally simple and are documented so the notebook remains transparent and editable.

## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from AlphaSimPy import runMacs, SimParam, newPop, randCross, setPheno, selectInd, meanG, varG

np.random.seed(123)
print('AlphaSimPy BRAID conversion notebook')
print('Libraries imported successfully.')

## Global Parameters

These parameters are derived from the BRAID abstraction where possible, with explicit defaults for missing values.

In [ ]:
# BRAID-derived horizon
n_cycles = 2

# Genome parameters from BRAID
n_chr = 10
n_qtl = 100
n_snp = 0
ploidy = 2

# Assumed founder and breeding sizes
n_founders_per_pool = 60
n_lines_per_pool = 30
n_testers = 3
n_recomb_crosses = 20
n_selected_hybrids = 18  # ~10% if 180 hybrids are evaluated
n_selected_parents_per_pool = 12

# Trait and phenotype settings
trait_mean = 0.0
trait_var = 1.0
heritability = 0.3
var_e = trait_var * (1 - heritability) / heritability

print('Simulation Parameters:')
print(f'  Cycles: {n_cycles}')
print(f'  Chromosomes: {n_chr}')
print(f'  QTL per chromosome: {n_qtl}')
print(f'  Founders per pool: {n_founders_per_pool}')
print(f'  Derived lines per pool: {n_lines_per_pool}')
print(f'  Testers per side: {n_testers}')
print(f'  Selected hybrids: {n_selected_hybrids}')
print(f'  Selected parents per pool: {n_selected_parents_per_pool}')
print(f'  Error variance: {var_e:.3f}')

## Create Founder Populations

The BRAID program begins with two C0 heterotic pools, A and B. We simulate a single founder population and split it into two pools.

In [ ]:
print('Creating founder populations...')

founder_pop = runMacs(
    nInd=n_founders_per_pool * 2,
    nChr=n_chr,
    segSites=n_qtl + n_snp,
    inbred=True,
    species='MAIZE'
)

SP = SimParam(founder_pop)
SP.addTraitAG(nQtlPerChr=n_qtl, mean=trait_mean, var=trait_var)
SP.setVarE(varE=var_e)

c0_pool_a = newPop(founder_pop[0:n_founders_per_pool], simParam=SP)
c0_pool_b = newPop(founder_pop[n_founders_per_pool:(2 * n_founders_per_pool)], simParam=SP)

print(f'C0 Pool A size: {c0_pool_a.n_ind}')
print(f'C0 Pool B size: {c0_pool_b.n_ind}')
print(f'Mean G A: {meanG(c0_pool_a)[0]:.3f}')
print(f'Mean G B: {meanG(c0_pool_b)[0]:.3f}')
print(f'Var G A: {varG(c0_pool_a)[0]:.3f}')
print(f'Var G B: {varG(c0_pool_b)[0]:.3f}')

## Helper Functions

The following helper functions keep the cycle simulation readable. They implement a simple approximation of line derivation, tester sampling, testcross evaluation, and recombination.

In [ ]:
def deriveLines(pool, nLines, varE, simParam):
    progeny = randCross(pool, nCrosses=nLines, simParam=simParam)
    progeny = setPheno(progeny, varE=varE, simParam=simParam)
    lines = selectInd(progeny, nInd=nLines, use='pheno', simParam=simParam)
    return lines

def chooseTesters(pool, nTesters, simParam):
    testers = selectInd(pool, nInd=nTesters, use='gv', simParam=simParam)
    return testers

def makeFactorialTestcrosses(lines, testers, simParam):
    hybrid_pops = []
    parent_map = []
    for line_idx in range(lines.n_ind):
        line_pop = selectInd(lines, nInd=1, parents=[line_idx], simParam=simParam)
        for tester_idx in range(testers.n_ind):
            tester_pop = selectInd(testers, nInd=1, parents=[tester_idx], simParam=simParam)
            hybrid = randCross(line_pop, nCrosses=1, parents2=tester_pop, simParam=simParam)
            hybrid_pops.append(hybrid)
            parent_map.append((line_idx, tester_idx))
    return hybrid_pops, parent_map

def mergeHybridSets(hybrid_pops):
    merged = hybrid_pops[0]
    for hp in hybrid_pops[1:]:
        merged = merged.concat(hp)
    return merged


## Run the BRAID-Inspired Breeding Cycles

Each cycle follows the BRAID workflow:
1. derive lines from each C0/C1 pool
2. choose three opposite-side testers
3. create testcross hybrids
4. evaluate hybrids phenotypically
5. select best hybrids for product development
6. identify contributing female and male parents
7. recombine selected parents to form the next heterotic pools

In [ ]:
results = []

pool_a = c0_pool_a
pool_b = c0_pool_b
released_varieties = None
product_development_material = None

for cycle in range(1, n_cycles + 1):
    print(f'\n=== Cycle {cycle} ===')
    
    l_lines_a = deriveLines(pool_a, n_lines_per_pool, var_e, SP)
    l_lines_b = deriveLines(pool_b, n_lines_per_pool, var_e, SP)
    
    testers_a = chooseTesters(pool_a, n_testers, SP)
    testers_b = chooseTesters(pool_b, n_testers, SP)
    
    ho_a_list, map_a = makeFactorialTestcrosses(l_lines_a, testers_b, SP)
    ho_b_list, map_b = makeFactorialTestcrosses(l_lines_b, testers_a, SP)
    
    all_hybrids = ho_a_list + ho_b_list
    selected_hybrid_records = []
    hybrid_scores = []
    
    for i, hybrid in enumerate(all_hybrids):
        hybrid = setPheno(hybrid, varE=var_e, simParam=SP)
        score = float(meanG(hybrid)[0])
        hybrid_scores.append((i, score, hybrid))
    
    hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)
    top_hybrids = hybrid_scores[:n_selected_hybrids]
    
    female_parent_indices = []
    male_parent_indices = []
    
    for idx, score, hybrid in top_hybrids:
        if idx < len(map_a):
            line_idx, tester_idx = map_a[idx]
            female_parent_indices.append(line_idx)
        else:
            adj = idx - len(map_a)
            line_idx, tester_idx = map_b[adj]
            male_parent_indices.append(line_idx)
        selected_hybrid_records.append(score)
    
    female_parent_indices = sorted(list(set(female_parent_indices)))
    male_parent_indices = sorted(list(set(male_parent_indices)))
    
    if len(female_parent_indices) == 0:
        female_parent_indices = list(range(min(n_selected_parents_per_pool, l_lines_a.n_ind)))
    if len(male_parent_indices) == 0:
        male_parent_indices = list(range(min(n_selected_parents_per_pool, l_lines_b.n_ind)))
    
    female_parent_indices = female_parent_indices[:n_selected_parents_per_pool]
    male_parent_indices = male_parent_indices[:n_selected_parents_per_pool]
    
    selected_female_parents = selectInd(l_lines_a, nInd=len(female_parent_indices), parents=female_parent_indices, simParam=SP)
    selected_male_parents = selectInd(l_lines_b, nInd=len(male_parent_indices), parents=male_parent_indices, simParam=SP)
    
    c1_pool_a = randCross(selected_female_parents, nCrosses=n_recomb_crosses, simParam=SP)
    c1_pool_b = randCross(selected_male_parents, nCrosses=n_recomb_crosses, simParam=SP)
    
    product_development_material = [item[2] for item in top_hybrids]
    released_varieties = product_development_material[:max(1, min(5, len(product_development_material)))]
    
    cycle_result = {
        'cycle': cycle,
        'poolA_meanG': float(meanG(pool_a)[0]),
        'poolB_meanG': float(meanG(pool_b)[0]),
        'poolA_varG': float(varG(pool_a)[0]),
        'poolB_varG': float(varG(pool_b)[0]),
        'linesA_meanG': float(meanG(l_lines_a)[0]),
        'linesB_meanG': float(meanG(l_lines_b)[0]),
        'bestHybridMeanG': float(np.mean([x[1] for x in top_hybrids])),
        'nSelectedFemaleParents': int(len(female_parent_indices)),
        'nSelectedMaleParents': int(len(male_parent_indices)),
        'nReleasedVarieties': int(len(released_varieties))
    }
    results.append(cycle_result)
    
    print(f"Derived lines A: {l_lines_a.n_ind}, lines B: {l_lines_b.n_ind}")
    print(f"Top hybrid mean G: {cycle_result['bestHybridMeanG']:.3f}")
    print(f"Selected female parents: {cycle_result['nSelectedFemaleParents']}")
    print(f"Selected male parents: {cycle_result['nSelectedMaleParents']}")
    print(f"Released varieties placeholder count: {cycle_result['nReleasedVarieties']}")
    
    pool_a = c1_pool_a
    pool_b = c1_pool_b

results_df = pd.DataFrame(results)
print('\nCycle summary table:')
print(results_df)

## Results Summary

The table below summarizes the recurrent selection cycles and the transition from C0 to C1 pools.

In [ ]:
results_df

## Plot Genetic Trends

We track the genetic mean and variance requested in the BRAID outputs section.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(results_df['cycle'], results_df['poolA_meanG'], marker='o', label='Pool A meanG')
axes[0].plot(results_df['cycle'], results_df['poolB_meanG'], marker='s', label='Pool B meanG')
axes[0].plot(results_df['cycle'], results_df['bestHybridMeanG'], marker='^', label='Best hybrid meanG')
axes[0].set_title('Genetic Mean by Cycle')
axes[0].set_xlabel('Cycle')
axes[0].set_ylabel('Mean Genetic Value')
axes[0].legend()

axes[1].plot(results_df['cycle'], results_df['poolA_varG'], marker='o', label='Pool A varG')
axes[1].plot(results_df['cycle'], results_df['poolB_varG'], marker='s', label='Pool B varG')
axes[1].set_title('Genetic Variance by Cycle')
axes[1].set_xlabel('Cycle')
axes[1].set_ylabel('Genetic Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

## Final Notes

This notebook is a faithful but simplified executable interpretation of the BRAID abstraction.

Potential future refinements include:
- explicit doubled haploid or selfing pipelines for line derivation
- more realistic hybrid creation functions if available in the installed AlphaSimPy version
- explicit inbreeding tracking
- multi-environment and replicated phenotyping
- genomic prediction once the BRAID abstraction specifies a model